In [1]:
# Install RAPIDS correctly in Google Colab
!git clone https://github.com/rapidsai/rapidsai-csp-utils.git
!python rapidsai-csp-utils/colab/rapids-colab.py

Cloning into 'rapidsai-csp-utils'...
remote: Enumerating objects: 629, done.
remote: Counting objects: 100% (195/195), done.
remote: Compressing objects: 100% (110/110), done.
remote: Total 629 (delta 149), reused 86 (delta 85), pack-reused 434 (from 3)
Receiving objects: 100% (629/629), 207.69 KiB | 2.88 MiB/s, done.
Resolving deltas: 100% (323/323), done.
python3: can't open file '/content/rapidsai-csp-utils/colab/rapids-colab.py': [Errno 2] No such file or directory


In [2]:
!nvidia-smi

Wed Mar 18 18:12:45 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   37C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [3]:
# Install RAPIDS
!pip install -q cudf-cu12 cupy-cuda12x --extra-index-url=https://pypi.nvidia.com


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.1/40.1 MB 185.9 MB/s eta 0:00:00


In [4]:
import cudf
import pandas as pd
import plotly.express as px

In [5]:
print("RAPIDS version:", cudf.__version__)

RAPIDS version: 26.02.01


## Load the Dataset

In [6]:
# Mount Drive instead of csv upload

from google.colab import drive
drive.mount('/content/drive')



Mounted at /content/drive


In [11]:
%%time

gdf = cudf.read_csv(
    "/content/drive/MyDrive/fire_data/fire_archive_J1V-C2_726206.csv"
)

CPU times: user 1.12 s, sys: 957 ms, total: 2.08 s
Wall time: 1.96 s


In [8]:

# Preview
gdf.head()

,latitude,longitude,brightness,scan,track,acq_date,acq_time,satellite,instrument,confidence,version,bright_t31,frp,daynight,type
0,70.94708,74.53897,331.97,0.54,0.51,2024-02-01,7,N20,VIIRS,n,2,246.54,3.93,N,0
1,68.41541,83.62364,367.00,0.50,0.66,2024-02-01,7,N20,VIIRS,h,2,249.76,7.27,N,2
2,66.66642,80.44739,367.00,0.59,0.70,2024-02-01,7,N20,VIIRS,h,2,249.78,13.21,N,2
3,70.99835,73.84378,345.58,0.53,0.50,2024-02-01,7,N20,VIIRS,n,2,252.52,23.67,N,0
4,67.28266,83.03193,329.51,0.58,0.70,2024-02-01,7,N20,VIIRS,n,2,251.59,3.59,N,2


In [12]:
gdf.columns

Index(['latitude', 'longitude', 'brightness', 'scan', 'track', 'acq_date',
       'acq_time', 'satellite', 'instrument', 'confidence', 'version',
       'bright_t31', 'frp', 'daynight', 'type'],
      dtype='object')

In [ ]:
gdf.describe()

,latitude,longitude,brightness,scan,track,acq_time,version,bright_t31,frp,type
count,2.262093e+07,2.262093e+07,2.262093e+07,2.262093e+07,2.262093e+07,2.262093e+07,2.262093e+07,2.262093e+07,2.262093e+07,2.262093e+07
mean,6.005813e+00,1.640374e+01,3.340767e+02,4.600690e-01,4.810481e-01,1.203778e+03,2.000000e+00,2.992271e+02,9.720888e+00,1.322104e-01
std,2.397649e+01,6.618323e+01,1.747975e+01,8.747307e-02,1.151069e-01,5.498950e+02,1.344141e-16,1.091392e+01,2.057355e+01,5.151020e-01
min,-8.585243e+01,-1.798427e+02,2.079300e+02,3.200000e-01,3.600000e-01,0.000000e+00,2.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00
25%,-1.263833e+01,-4.337048e+01,3.278100e+02,3.900000e-01,3.800000e-01,8.580000e+02,2.000000e+00,2.921300e+02,2.570000e+00,0.000000e+00
50%,1.475990e+00,2.309261e+01,3.367600e+02,4.400000e-01,4.500000e-01,1.154000e+03,2.000000e+00,2.999500e+02,5.000000e+00,0.000000e+00
75%,1.771371e+01,4.023370e+01,3.447500e+02,5.100000e-01,5.600000e-01,1.642000e+03,2.000000e+00,3.064100e+02,9.430000e+00,0.000000e+00
max,8.036205e+01,1.799608e+02,3.670000e+02,8.000000e-01,7.800000e-01,2.359000e+03,2.000000e+00,3.811700e+02,3.276550e+03,3.000000e+00


In [13]:
gdf.isnull().sum()

latitude      0
longitude     0
brightness    0
scan          0
track         0
acq_date      0
acq_time      0
satellite     0
instrument    0
confidence    0
version       0
bright_t31    0
frp           0
daynight      0
type          0
dtype: int64

In [14]:
gdf.duplicated().sum()

np.int64(0)

In [15]:
gdf.shape

(22620926, 15)

## Basic Data Processing

In [10]:
process_start = time.time()

gdf['acq_date'] = cudf.to_datetime(gdf['acq_date'])

fires_daily = (
    gdf.groupby('acq_date')
    .size()
    .reset_index(name='fire_count')
)

process_time = time.time() - process_start
print(f"GPU Time: {process_time:.2f} seconds")

GPU Time: 0.19 seconds


In [16]:
gdf['acq_date'] = cudf.to_datetime(gdf['acq_date'])
gdf['month'] = gdf['acq_date'].dt.month

In [17]:
# sanity check
min_month = gdf['month'].min()
max_month = gdf['month'].max()

print(f"The dataset contains data from month {min_month} to month {max_month}.")
print(f"This covers a total of {max_month - min_month + 1} months.")

The dataset contains data from month 1 to month 12.
This covers a total of 12 months.


In [18]:
gdf['acq_date'].min(), gdf['acq_date'].max()

(np.datetime64('2024-02-01T00:00:00.000000000'),
 np.datetime64('2025-02-01T00:00:00.000000000'))

In [19]:
gdf['acq_date'].dt.year.value_counts().sort_index()

acq_date
2024    20850757
2025     1770169
Name: count, dtype: int64

### Feature engineering "period"

In [20]:
# Feature Engineering
gdf['period'] = "Dec–Jan"

gdf.loc[gdf['month'].isin([2,3]), 'period'] = "Feb–Mar"
gdf.loc[gdf['month'].isin([4,5]), 'period'] = "Apr–May"
gdf.loc[gdf['month'].isin([6,7]), 'period'] = "Jun–Jul"
gdf.loc[gdf['month'].isin([8,9]), 'period'] = "Aug–Sep"
gdf.loc[gdf['month'].isin([10,11]), 'period'] = "Oct–Nov"

## Create Data for Dashboard

### Fires Over Time (Daily Trend)

In [21]:
# print performance with RAPIDS
%%timeit

fires_over_time = (
    gdf.groupby("acq_date")
    .size()
    .reset_index(name="fire_count")
)


29.6 ms ± 136 µs per loop (mean ± std. dev. of 7 runs, 10 loops each)


In [22]:
fires_over_time = (
    gdf.groupby("acq_date")
    .size()
    .reset_index(name="fire_count")
)
fires_over_time = fires_over_time.sort_values("acq_date")
fires_over_time.head()

,acq_date,fire_count
97,2024-02-01,40731
148,2024-02-02,49132
51,2024-02-03,45787
341,2024-02-05,4520
296,2024-02-06,67008


In [23]:
# Convert to pandas
fires_over_time_pd = fires_over_time.to_pandas()

### Fires by Seasonal Period

In [24]:
gdf['period'].value_counts()

period
Aug–Sep    6296398
Jun–Jul    4576940
Dec–Jan    3167854
Apr–May    2910139
Feb–Mar    2849462
Oct–Nov    2820133
Name: count, dtype: int64

In [25]:
period_order = [
    "Feb–Mar",
    "Apr–May",
    "Jun–Jul",
    "Aug–Sep",
    "Oct–Nov",
    "Dec–Jan"
]

In [26]:
fires_by_period = (
    gdf.groupby("period")
    .size()
    .reset_index(name="fire_count")
)
fires_by_period.head(10)

,period,fire_count
0,Feb–Mar,2849462
1,Dec–Jan,3167854
2,Apr–May,2910139
3,Jun–Jul,4576940
4,Oct–Nov,2820133
5,Aug–Sep,6296398


In [27]:
%%timeit

fires_by_period = (
    gdf.groupby("period")
    .size()
    .reset_index(name="fire_count")
)
fires_by_period.head(10)

24.4 ms ± 232 µs per loop (mean ± std. dev. of 7 runs, 10 loops each)


In [28]:
# order periods in chronological order

fires_by_period["period"] = fires_by_period["period"].astype("category")
fires_by_period["period"] = fires_by_period["period"].cat.set_categories(
    period_order, ordered=True
)

fires_by_period = fires_by_period.sort_values("period")
fires_by_period.head(6)

,period,fire_count
0,Feb–Mar,2849462
2,Apr–May,2910139
3,Jun–Jul,4576940
5,Aug–Sep,6296398
4,Oct–Nov,2820133
1,Dec–Jan,3167854


In [29]:
#convert to pandas
fires_by_period_pd = fires_by_period.to_pandas()

### First Intensity Stats

In [30]:
frp_stats = (
    gdf.groupby("period")
    .agg({
        "frp": ["mean","max"]
    })
)

frp_stats.head(6)

frp         
              mean      max
period                     
Feb–Mar   8.356836  1603.07
Dec–Jan   8.041774  2966.53
Apr–May   9.313389  2160.96
Jun–Jul  10.448896  2825.18
Oct–Nov   9.365573  3276.55
Aug–Sep  11.001281  2257.42

Reoganizing the data...

In [31]:
frp_stats.columns = ["mean_frp", "max_frp"]
frp_stats = frp_stats.reset_index()

In [32]:
frp_stats_pd = frp_stats.to_pandas()

### Spatial Hotspots
Issue - when first plotting this, the count is 1 because the grouping by the exact coordinatre pairs makes each area very small.

In [33]:
spatial_counts = (
    gdf.groupby(["latitude","longitude"])
    .size()
    .reset_index(name="fire_count")
)

spatial_counts.head(10)

,latitude,longitude,fire_count
0,-17.32117,-59.05849,1
1,48.84703,-116.81227,1
2,9.02345,-0.63694,1
3,-16.32758,29.44296,1
4,36.71181,43.79346,1
5,45.35575,30.89752,1
6,-6.61288,29.05069,1
7,31.49057,75.21890,1
8,-7.36979,15.65558,1
9,-15.08901,-62.86617,1


Fix: group by spatial grid cells

In [34]:
gdf["lat_bin"] = gdf["latitude"].round(1)
gdf["lon_bin"] = gdf["longitude"].round(1)

In [35]:
spatial_counts = (
    gdf.groupby(["lat_bin", "lon_bin"])
    .size()
    .reset_index(name="fire_count")
)
# only render the top hotspots to speed up pandas processing
spatial_counts = spatial_counts.sort_values("fire_count", ascending=False).head(500)
spatial_counts.head(15)

,lat_bin,lon_bin,fire_count
51193,27.5,52.6,9837
289909,-1.4,29.2,9176
65629,27.7,52.2,8242
4633,63.9,-22.4,7587
370439,9.6,-63.6,7541
96779,30.6,47.3,6984
291261,23.8,86.4,6626
301045,30.2,47.4,6404
138554,9.7,-63.5,6260
190129,31.0,47.3,6003


In [36]:
%%timeit

spatial_counts = (
    gdf.groupby(["lat_bin", "lon_bin"])
    .size()
    .reset_index(name="fire_count")
)
# only render the top hotspots to speed up pandas processing
spatial_counts = spatial_counts.sort_values("fire_count", ascending=False).head(500)
spatial_counts.head(15)

49.2 ms ± 200 µs per loop (mean ± std. dev. of 7 runs, 10 loops each)


In [37]:
spatial_counts_pd = spatial_counts.to_pandas()

### Confidence Distribution

In [38]:
%%time

confidence_counts = (
    gdf.groupby("confidence")
    .size()
    .reset_index(name="count")
)


CPU times: user 23.8 ms, sys: 7.97 ms, total: 31.8 ms
Wall time: 31.4 ms


In [39]:
confidence_counts = (
    gdf.groupby("confidence")
    .size()
    .reset_index(name="count")
)
confidence_counts = confidence_counts.sort_values("count", ascending=False)
confidence_counts.head()

,confidence,count
0,n,18920429
2,l,2570439
1,h,1130058


In [40]:
# convert to pandas
confidence_counts_pd = confidence_counts.to_pandas()

In [41]:
frp_by_confidence = (
    gdf.groupby("confidence")
    .agg({
        "frp": ["mean", "max"]
    })
)
frp_by_confidence.columns = ["mean_frp", "max_frp"]
frp_by_confidence = frp_by_confidence.reset_index()
frp_by_confidence.head()

,confidence,mean_frp,max_frp
0,n,8.172368,2216.31
1,h,27.738015,2825.18
2,l,13.198216,3276.55


In [42]:
# convert to pandas
frp_by_confidence_pd = frp_by_confidence.to_pandas()

# Save aggregated data frames

In [43]:
base_path = "/content/drive/MyDrive/fire_data/"

fires_over_time_pd.to_csv(base_path + "fires_over_time.csv", index=False)

fires_by_period_pd.to_csv(base_path + "fires_by_period.csv", index=False)

frp_stats_pd.to_csv(base_path + "frp_stats.csv", index=False)

spatial_counts_pd.to_csv(base_path + "spatial_counts.csv", index=False)

confidence_counts_pd.to_csv(base_path + "confidence_counts.csv", index=False)

frp_by_confidence_pd.to_csv(base_path + "frp_by_confidence.csv", index=False)


In [ ]:
base_path = "/content/drive/MyDrive/fire_data/"

columns_needed = [
    "acq_date",
    "latitude",
    "longitude",
    "frp",
    "confidence",
    "period"
]

filtered_df = gdf[columns_needed].to_pandas()

filtered_df.to_csv(base_path + "dashboard_dataset.csv", index=False)


KeyboardInterrupt: 

In [ ]:
df = pd.read_csv(base_path + "dashboard_dataset.csv")
df["acq_date"] = pd.to_datetime(df["acq_date"])


KeyboardInterrupt: 

### Code to run dataframes saved from previous GPU run -

In [ ]:
import pandas as pd

from google.colab import drive
drive.mount('/content/drive')

base_path = "/content/drive/MyDrive/fire_data/"

fires_over_time_pd = pd.read_csv(base_path + "fires_over_time.csv")
fires_by_period_pd = pd.read_csv(base_path + "fires_by_period.csv")
frp_stats_pd = pd.read_csv(base_path + "frp_stats.csv")
spatial_counts_pd = pd.read_csv(base_path + "spatial_counts.csv")
confidence_counts_pd = pd.read_csv(base_path + "confidence_counts.csv")
frp_by_confidence_pd = pd.read_csv(base_path + "frp_by_confidence.csv")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


# Plotly Visualizations

In [ ]:
# Sanity check
frp_by_confidence_pd.head()

,confidence,mean_frp,max_frp
0,n,8.172368,2216.31
1,h,27.738015,2825.18
2,l,13.198216,3276.55


## Temporal Wildfire Patterns

In [ ]:
import plotly.express as px

fig_time = px.line(
    fires_over_time_pd,
    x="acq_date",
    y="fire_count",
    title="Wildfire Detections Over Time",
    labels={
        "acq_date": "Date",
        "fire_count": "Number of Fires"
    }
)

fig_time.show()

## Seasonal Fire Counts

In [ ]:
fig_period = px.bar(
    fires_by_period_pd,
    x="period",
    y="fire_count",
    title="Wildfire Activity by Season",
    labels={
        "period": "Season",
        "fire_count": "Fire Count"
    }
)

fig_period.show()

## Fire Intensity by Season

In [ ]:
fig_frp = px.bar(
    frp_stats_pd,
    x="period",
    y="mean_frp",
    title="Average Fire Intensity by Season",
    labels={
        "period": "Season",
        "mean_frp": "Average FRP"
    }
)

fig_frp.show()

## First Hotspot Map

In [ ]:
fig_map = px.scatter_mapbox(
    spatial_counts_pd,
    lat="lat_bin",
    lon="lon_bin",
    size="fire_count",
    color="fire_count",
    zoom=2,
    mapbox_style="open-street-map",
    title="Global Wildfire Hotspots"
)

fig_map.show()